In [1]:
# %% [markdown]
# # Descobrir CEPs válidos em Ubatuba
# Consulta ViaCEP a partir de CEPs aleatórios na faixa da cidade e salva os válidos em PKL.

# %%
import os
import time
import pickle
import random
import requests

ARQUIVO_SAIDA = "ceps_ubatuba.pkl"
META = 200
PAUSA = 0.3
MAX_TENTATIVAS = 3000

FAIXAS_CEP = [
    (11680, 11681),
    (11682, 11683),
    (11684, 11685),
    (11686, 11687),
    (11688, 11689),
    (11690, 11691),
    (11692, 11693),
    (11694, 11695),
    (11696, 11697),
    (11698, 11699),
]

# %%
def gerar_cep_aleatorio():
    inicio, fim = random.choice(FAIXAS_CEP)
    prefixo = random.randint(inicio, fim)
    sufixo = random.randint(0, 999)
    return f"{prefixo}{sufixo:03d}"

def consultar_viacep(cep):
    url = f"https://viacep.com.br/ws/{cep}/json/"
    try:
        r = requests.get(url, timeout=5)
        if r.status_code != 200:
            return None
        data = r.json()
        if data.get("erro"):
            return None
        if data.get("localidade", "").lower() != "ubatuba":
            return None
        return {
            "cep": data.get("cep", "").replace("-", ""),
            "logradouro": data.get("logradouro", ""),
            "bairro": data.get("bairro", ""),
            "cidade": data.get("localidade", ""),
            "estado": data.get("uf", ""),
        }
    except Exception:
        return None

# %%
encontrados = []
ceps_usados = set()
tentativas = 0

while len(encontrados) < META and tentativas < MAX_TENTATIVAS:
    tentativas += 1
    cep = gerar_cep_aleatorio()

    if cep in ceps_usados:
        continue
    ceps_usados.add(cep)

    resultado = consultar_viacep(cep)

    if resultado:
        encontrados.append(resultado)
        print(f"[{len(encontrados):3d}/{META}] OK  {resultado['cep']} - {resultado['logradouro']} - {resultado['bairro']}")
    else:
        if tentativas % 50 == 0:
            print(f"[tentativa {tentativas}] {len(encontrados)}/{META}")

    time.sleep(PAUSA)

print(f"\nTotal: {len(encontrados)} CEPs válidos em Ubatuba")

# %%
with open(ARQUIVO_SAIDA, "wb") as f:
    pickle.dump(encontrados, f)

print(f"Salvo em: {ARQUIVO_SAIDA}")

# %%
# Ver amostra
encontrados[:10]

[  1/200] OK  11695216 - Rua do Balmoral - Perequê Açu
[  2/200] OK  11682492 - Rua Paraíba - Lagoinha
[  3/200] OK  11689234 - Rua Bonsucesso - Estufa I
[  4/200] OK  11697426 - Vereda 6 - Promirim
[  5/200] OK  11681050 - Rua Benedita Luiza dos Santos - Comunidade do Quilombo de Caçandoca
[  6/200] OK  11686506 - Rua Margarida Cabral dos Santos - Perequê Mirim
[  7/200] OK  11680971 - Rua Aparecida Santos Velloso - Centro
[  8/200] OK  11694518 - Rua Golfinho - Ressaca
[  9/200] OK  11693130 - Rua dos Jeribás - Ipiranguinha
[tentativa 100] 9/200
[ 10/200] OK  11696522 - Rua 30 - Itamambuca
[ 11/200] OK  11689306 - Rua 8 - Estufa II
[ 12/200] OK  11681574 - Rua Araponga - Sertão da Quina
[ 13/200] OK  11697428 - Vereda 7 - Promirim
[ 14/200] OK  11690428 - Rua Mato Grosso - Umuarama
[ 15/200] OK  11686767 - Rua Sagitário - Enseada
[ 16/200] OK  11689608 - Rua Saveiros - Sesmaria
[ 17/200] OK  11685446 - Rua Ametista - Lázaro
[ 18/200] OK  11693093 - Rua da Fé - Ipiranguinha
[tentativa

[{'cep': '11695216',
  'logradouro': 'Rua do Balmoral',
  'bairro': 'Perequê Açu',
  'cidade': 'Ubatuba',
  'estado': 'SP'},
 {'cep': '11682492',
  'logradouro': 'Rua Paraíba',
  'bairro': 'Lagoinha',
  'cidade': 'Ubatuba',
  'estado': 'SP'},
 {'cep': '11689234',
  'logradouro': 'Rua Bonsucesso',
  'bairro': 'Estufa I',
  'cidade': 'Ubatuba',
  'estado': 'SP'},
 {'cep': '11697426',
  'logradouro': 'Vereda 6',
  'bairro': 'Promirim',
  'cidade': 'Ubatuba',
  'estado': 'SP'},
 {'cep': '11681050',
  'logradouro': 'Rua Benedita Luiza dos Santos',
  'bairro': 'Comunidade do Quilombo de Caçandoca',
  'cidade': 'Ubatuba',
  'estado': 'SP'},
 {'cep': '11686506',
  'logradouro': 'Rua Margarida Cabral dos Santos',
  'bairro': 'Perequê Mirim',
  'cidade': 'Ubatuba',
  'estado': 'SP'},
 {'cep': '11680971',
  'logradouro': 'Rua Aparecida Santos Velloso',
  'bairro': 'Centro',
  'cidade': 'Ubatuba',
  'estado': 'SP'},
 {'cep': '11694518',
  'logradouro': 'Rua Golfinho',
  'bairro': 'Ressaca',
  'cid